In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings
sales= pd.read_csv("sales_date.csv")
sales.dropna(inplace= True)
sales.isnull().sum()

order_id                0
customer_id             0
product_id              0
product_count           0
product_price           0
product_discount        0
price_after_discount    0
order_date              0
dtype: int64

In [2]:
sales.shape

(1494806, 8)

In [3]:
sales.groupby('customer_id').size().sort_values(ascending=False).reset_index(name='order_count').head()

,customer_id,order_count
0,40880,36
1,72034,32
2,24271,32
3,97967,32
4,94904,32


# Feature Creation Cus*order_date ---------- Product ------  

In [4]:
df = sales#.sample(100)#[sales["customer_id"] == 24271].copy()
df["order_date"] = pd.to_datetime(df["order_date"])
df["day_of_week"] = df["order_date"].dt.dayofweek
df["day_of_month"] = df["order_date"].dt.day
df["month"] = df["order_date"].dt.month
df["quarter"] = df["order_date"].dt.quarter
df["week_of_year"] = df["order_date"].dt.isocalendar().week
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

In [5]:
df['spend'] =df['price_after_discount']*df['product_count']
df.head()

,order_id,customer_id,product_id,product_count,product_price,product_discount,price_after_discount,order_date,day_of_week,day_of_month,month,quarter,week_of_year,is_weekend,spend
0,6,28663,413,8,95.4065,5,90.64,2026-02-07,5,7,2,1,6,1,725.12
1,7,46674,370,12,21.8806,10,19.69,2026-03-02,0,2,3,1,10,0,236.28
2,15,19310,134,5,39.8545,6,37.46,2026-01-26,0,26,1,1,5,0,187.30
3,16,91867,364,24,8.8429,28,6.37,2026-04-30,3,30,4,2,18,0,152.88
4,17,14612,185,4,48.4195,19,39.22,2026-01-20,1,20,1,1,4,0,156.88


In [6]:
df.shape

(1494806, 15)

In [7]:
import pandas as pd
import numpy as np

# Keep only required columns
cols = [
    "customer_id", "order_date", "order_id", "product_id",
    "product_count", "spend", "price_after_discount",
    "product_discount", "product_price"
]

df2 = df[cols].copy()

group_cols = ["customer_id", "order_date"]

# -----------------------------------
# 1. Create groupby ONCE
# -----------------------------------
g = df2.groupby(group_cols, sort=False, observed=True)


# -----------------------------------
# 2. Basic aggregations
# -----------------------------------
cpd = g.agg(
    purchase_count=("order_id", "count"),

    # Quantity
    avg_quantity=("product_count", "mean"),
    max_quantity=("product_count", "max"),

    # Spend
    total_spend=("spend", "sum"),
    avg_spend=("spend", "mean"),
    min_spend=("spend", "min"),
    max_spend=("spend", "max"),

    # Price after discount
    total_pad=("price_after_discount", "sum"),
    avg_pad=("price_after_discount", "mean"),
    min_pad=("price_after_discount", "min"),
    max_pad=("price_after_discount", "max"),

    # Product count
    total_pc=("product_count", "sum"),
    avg_pc=("product_count", "mean"),
    min_pc=("product_count", "min"),
    max_pc=("product_count", "max"),

    # Product discount
    total_pd=("product_discount", "sum"),
    avg_pd=("product_discount", "mean"),
    min_pd=("product_discount", "min"),
    max_pd=("product_discount", "max"),

    # Product price
    total_pp=("product_price", "sum"),
    avg_pp=("product_price", "mean"),
    min_pp=("product_price", "min"),
    max_pp=("product_price", "max"),
)


# -----------------------------------
# 3. Calculate ALL quantiles together
# -----------------------------------
quantile_cols = [
    "product_count",
    "spend",
    "price_after_discount",
    "product_discount",
    "product_price"
]

qs = [0.25, 0.50, 0.75, 0.90, 0.95]

quantiles = (
    g[quantile_cols]
    .quantile(qs)
    .unstack(level=-1)
)


# -----------------------------------
# 4. Rename quantile columns
# -----------------------------------
prefix = {
    "product_count": "pc",
    "spend": "spend",
    "price_after_discount": "pad",
    "product_discount": "pd",
    "product_price": "pp"
}

quantiles.columns = [
    f"p{int(q * 100)}_{prefix[col]}"
    for col, q in quantiles.columns
]


# Quantity names from your original code
quantiles["p25_quantity"] = quantiles["p25_pc"]
quantiles["p50_quantity"] = quantiles["p50_pc"]
quantiles["p75_quantity"] = quantiles["p75_pc"]
quantiles["p90_quantity"] = quantiles["p90_pc"]
quantiles["p95_quantity"] = quantiles["p95_pc"]


# -----------------------------------
# 5. Mode product
# -----------------------------------
mode_product = (
    df2.groupby(group_cols + ["product_id"], sort=False)
       .size()
       .rename("count")
       .reset_index()
       .sort_values("count", ascending=False)
       .drop_duplicates(group_cols)
       .set_index(group_cols)["product_id"]
       .rename("mode_product_id")
)


# -----------------------------------
# 6. Combine everything
# -----------------------------------
cpd = (
    cpd
    .join(quantiles)
    .join(mode_product)
    .reset_index()
)

print(cpd.shape)
print(cpd.head())

(1410027, 56)
   customer_id order_date  purchase_count  avg_quantity  max_quantity  \
0        28663 2026-02-07               2           8.0             8   
1        46674 2026-03-02               1          12.0            12   
2        19310 2026-01-26               1           5.0             5   
3        91867 2026-04-30               1          24.0            24   
4        14612 2026-01-20               1           4.0             4   

   total_spend  avg_spend  min_spend  max_spend  total_pad  ...   p50_pp  \
0      1242.72     621.36     517.60     725.12     155.34  ...  92.6319   
1       236.28     236.28     236.28     236.28      19.69  ...  21.8806   
2       187.30     187.30     187.30     187.30      37.46  ...  39.8545   
3       152.88     152.88     152.88     152.88       6.37  ...   8.8429   
4       156.88     156.88     156.88     156.88      39.22  ...  48.4195   

    p75_pp    p90_pp    p95_pp  p25_quantity  p50_quantity  p75_quantity  \
0  94.0192  94

In [8]:
columns = cpd.columns[3:]

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 100)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

Total columns: (1410027, 56)
['avg_quantity', 'max_quantity', 'total_spend', 'avg_spend', 'min_spend', 'max_spend', 'total_pad', 'avg_pad', 'min_pad', 'max_pad']
['total_pc', 'avg_pc', 'min_pc', 'max_pc', 'total_pd', 'avg_pd', 'min_pd', 'max_pd', 'total_pp', 'avg_pp']
['min_pp', 'max_pp', 'p25_pc', 'p50_pc', 'p75_pc', 'p90_pc', 'p95_pc', 'p25_spend', 'p50_spend', 'p75_spend']
['p90_spend', 'p95_spend', 'p25_pad', 'p50_pad', 'p75_pad', 'p90_pad', 'p95_pad', 'p25_pd', 'p50_pd', 'p75_pd']
['p90_pd', 'p95_pd', 'p25_pp', 'p50_pp', 'p75_pp', 'p90_pp', 'p95_pp', 'p25_quantity', 'p50_quantity', 'p75_quantity']
['p90_quantity', 'p95_quantity', 'mode_product_id']


In [9]:
cpd = cpd.sort_values( ["customer_id", "order_date"]).copy()
for i in cpd.columns:
    cpd[f"last_{i}"] = (cpd.groupby("customer_id")[i] .shift(1) )

In [10]:
columns = cpd.columns[3:]

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 100)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

Total columns: (1410027, 112)
['avg_quantity', 'max_quantity', 'total_spend', 'avg_spend', 'min_spend', 'max_spend', 'total_pad', 'avg_pad', 'min_pad', 'max_pad']
['total_pc', 'avg_pc', 'min_pc', 'max_pc', 'total_pd', 'avg_pd', 'min_pd', 'max_pd', 'total_pp', 'avg_pp']
['min_pp', 'max_pp', 'p25_pc', 'p50_pc', 'p75_pc', 'p90_pc', 'p95_pc', 'p25_spend', 'p50_spend', 'p75_spend']
['p90_spend', 'p95_spend', 'p25_pad', 'p50_pad', 'p75_pad', 'p90_pad', 'p95_pad', 'p25_pd', 'p50_pd', 'p75_pd']
['p90_pd', 'p95_pd', 'p25_pp', 'p50_pp', 'p75_pp', 'p90_pp', 'p95_pp', 'p25_quantity', 'p50_quantity', 'p75_quantity']
['p90_quantity', 'p95_quantity', 'mode_product_id', 'last_customer_id', 'last_order_date', 'last_purchase_count', 'last_avg_quantity', 'last_max_quantity', 'last_total_spend', 'last_avg_spend']
['last_min_spend', 'last_max_spend', 'last_total_pad', 'last_avg_pad', 'last_min_pad', 'last_max_pad', 'last_total_pc', 'last_avg_pc', 'last_min_pc', 'last_max_pc']
['last_total_pd', 'last_avg_pd

In [11]:
cpd.head()

,customer_id,order_date,purchase_count,avg_quantity,max_quantity,total_spend,avg_spend,min_spend,max_spend,total_pad,...,last_p50_pp,last_p75_pp,last_p90_pp,last_p95_pp,last_p25_quantity,last_p50_quantity,last_p75_quantity,last_p90_quantity,last_p95_quantity,last_mode_product_id
212589,1,2026-01-18,1,1.0,1,34.93,34.930,34.93,34.93,34.93,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
844071,1,2026-01-23,1,1.0,1,55.17,55.170,55.17,55.17,55.17,...,38.8096,38.8096,38.8096,38.8096,1.0,1.0,1.0,1.0,1.0,421.0
823362,1,2026-01-28,1,1.0,1,47.00,47.000,47.00,47.00,47.00,...,68.1097,68.1097,68.1097,68.1097,1.0,1.0,1.0,1.0,1.0,266.0
528976,1,2026-02-12,1,1.0,1,19.49,19.490,19.49,19.49,19.49,...,56.6209,56.6209,56.6209,56.6209,1.0,1.0,1.0,1.0,1.0,233.0
1135704,1,2026-02-16,2,1.0,1,62.91,31.455,27.60,35.31,62.91,...,21.4230,21.4230,21.4230,21.4230,1.0,1.0,1.0,1.0,1.0,139.0


In [12]:
cpd.columns

Index(['customer_id', 'order_date', 'purchase_count', 'avg_quantity',
       'max_quantity', 'total_spend', 'avg_spend', 'min_spend', 'max_spend',
       'total_pad',
       ...
       'last_p50_pp', 'last_p75_pp', 'last_p90_pp', 'last_p95_pp',
       'last_p25_quantity', 'last_p50_quantity', 'last_p75_quantity',
       'last_p90_quantity', 'last_p95_quantity', 'last_mode_product_id'],
      dtype='str', length=112)

In [13]:
cpd["Same_order"] = (cpd["mode_product_id"] == cpd["last_mode_product_id"]).astype(int)
cpd["Last_transaction_days"] = (cpd["order_date"] - cpd["last_order_date"] ).dt.days
cpd.drop(['last_customer_id','last_mode_product_id'],axis=1 , inplace = True)

In [14]:
cpd.columns

Index(['customer_id', 'order_date', 'purchase_count', 'avg_quantity',
       'max_quantity', 'total_spend', 'avg_spend', 'min_spend', 'max_spend',
       'total_pad',
       ...
       'last_p75_pp', 'last_p90_pp', 'last_p95_pp', 'last_p25_quantity',
       'last_p50_quantity', 'last_p75_quantity', 'last_p90_quantity',
       'last_p95_quantity', 'Same_order', 'Last_transaction_days'],
      dtype='str', length=112)

In [15]:
new_cols = [    'purchase_count', 'avg_quantity', 'max_quantity',
                #'p25_quantity','p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity',
                'total_spend', 'avg_spend', 'min_spend', 'max_spend',
                #'p25_spend','p50_spend', 'p75_spend', 'p90_spend', 'p95_spend',
                'total_pad','avg_pad', 'min_pad', 'max_pad', 
                #'p25_pad', 'p50_pad', 'p75_pad','p90_pad', 'p95_pad',
                'total_pc', 'avg_pc', 'min_pc', 'max_pc',
                #'p25_pc', 'p50_pc', 'p75_pc', 'p90_pc', 'p95_pc', 
                'total_pd', 'avg_pd','min_pd', 'max_pd', 
                #'p25_pd', 'p50_pd', 'p75_pd', 'p90_pd', 'p95_pd',
                'total_pp', 'avg_pp', 'min_pp', 'max_pp',
                #'p25_pp', 'p50_pp', 'p75_pp','p90_pp', 'p95_pp' 
           ]

In [ ]:
import pandas as pd
import numpy as np
import time

start = time.time()

# --------------------------------------------------
# 1. Prepare data ONCE
# --------------------------------------------------

group_cols = ["customer_id"]
windows = [2, 3, 7, 30]

cpd["order_date"] = pd.to_datetime(cpd["order_date"])

cpd = (
    cpd.sort_values(["customer_id", "order_date"])
       .reset_index(drop=True)
)

# --------------------------------------------------
# 2. Process each feature
# --------------------------------------------------

for col in new_cols:

    # Historical value only — prevents data leakage
    hist = cpd.groupby("customer_id", sort=False)[col].shift(1)

    # Temporary dataframe
    temp = pd.DataFrame({
        "customer_id": cpd["customer_id"].values,
        "order_date": cpd["order_date"].values,
        "value": hist.values
    })

    # --------------------------------------------------
    # 3. Calculate rolling statistics
    # --------------------------------------------------

    for days in windows:

        r = (
            temp.set_index("order_date")
                .groupby("customer_id", sort=False)["value"]
                .rolling(
                    f"{days}D",
                    min_periods=1,
                    closed="both"
                )
        )

        # Basic statistics
        cpd[f"{col}_{days}d_mean"] = r.mean().to_numpy()
        cpd[f"{col}_{days}d_min"]  = r.min().to_numpy()
        cpd[f"{col}_{days}d_max"]  = r.max().to_numpy()
        cpd[f"{col}_{days}d_std"]  = r.std().to_numpy()

        # Quantiles
        for q, name in [
            (0.25, "p25"),
            (0.50, "p50"),
            (0.75, "p75"),
            (0.90, "p90"),
            (0.95, "p95")
        ]:
            cpd[f"{col}_{days}d_{name}"] = (
                r.quantile(q).to_numpy()
            )

        del r

    del temp, hist

    print(
        f"Completed: {col} | "
        f"Time: {(time.time() - start):.1f}s"
    )

# --------------------------------------------------
# 4. Final sorting
# --------------------------------------------------

cpd = (
    cpd.sort_values(["customer_id", "order_date"])
       .reset_index(drop=True)
)

print("\nFinished!")
print(f"Total time: {time.time() - start:.2f} seconds")
print("Final shape:", cpd.shape)

In [16]:
columns = cpd.columns[3:]

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 100)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

Total columns: (1410027, 112)
['avg_quantity', 'max_quantity', 'total_spend', 'avg_spend', 'min_spend', 'max_spend', 'total_pad', 'avg_pad', 'min_pad', 'max_pad']
['total_pc', 'avg_pc', 'min_pc', 'max_pc', 'total_pd', 'avg_pd', 'min_pd', 'max_pd', 'total_pp', 'avg_pp']
['min_pp', 'max_pp', 'p25_pc', 'p50_pc', 'p75_pc', 'p90_pc', 'p95_pc', 'p25_spend', 'p50_spend', 'p75_spend']
['p90_spend', 'p95_spend', 'p25_pad', 'p50_pad', 'p75_pad', 'p90_pad', 'p95_pad', 'p25_pd', 'p50_pd', 'p75_pd']
['p90_pd', 'p95_pd', 'p25_pp', 'p50_pp', 'p75_pp', 'p90_pp', 'p95_pp', 'p25_quantity', 'p50_quantity', 'p75_quantity']
['p90_quantity', 'p95_quantity', 'mode_product_id', 'last_order_date', 'last_purchase_count', 'last_avg_quantity', 'last_max_quantity', 'last_total_spend', 'last_avg_spend', 'last_min_spend']
['last_max_spend', 'last_total_pad', 'last_avg_pad', 'last_min_pad', 'last_max_pad', 'last_total_pc', 'last_avg_pc', 'last_min_pc', 'last_max_pc', 'last_total_pd']
['last_avg_pd', 'last_min_pd', 'l

In [17]:
cpd.columns

Index(['customer_id', 'order_date', 'purchase_count', 'avg_quantity',
       'max_quantity', 'total_spend', 'avg_spend', 'min_spend', 'max_spend',
       'total_pad',
       ...
       'last_p75_pp', 'last_p90_pp', 'last_p95_pp', 'last_p25_quantity',
       'last_p50_quantity', 'last_p75_quantity', 'last_p90_quantity',
       'last_p95_quantity', 'Same_order', 'Last_transaction_days'],
      dtype='str', length=112)

In [18]:
columns = cpd.columns

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 10)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

Total columns: (1410027, 112)
['customer_id', 'order_date', 'purchase_count', 'avg_quantity', 'max_quantity', 'total_spend', 'avg_spend', 'min_spend', 'max_spend', 'total_pad']
['avg_pad', 'min_pad', 'max_pad', 'total_pc', 'avg_pc', 'min_pc', 'max_pc', 'total_pd', 'avg_pd', 'min_pd']
['max_pd', 'total_pp', 'avg_pp', 'min_pp', 'max_pp', 'p25_pc', 'p50_pc', 'p75_pc', 'p90_pc', 'p95_pc']
['p25_spend', 'p50_spend', 'p75_spend', 'p90_spend', 'p95_spend', 'p25_pad', 'p50_pad', 'p75_pad', 'p90_pad', 'p95_pad']
['p25_pd', 'p50_pd', 'p75_pd', 'p90_pd', 'p95_pd', 'p25_pp', 'p50_pp', 'p75_pp', 'p90_pp', 'p95_pp']
['p25_quantity', 'p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity', 'mode_product_id', 'last_order_date', 'last_purchase_count', 'last_avg_quantity', 'last_max_quantity']
['last_total_spend', 'last_avg_spend', 'last_min_spend', 'last_max_spend', 'last_total_pad', 'last_avg_pad', 'last_min_pad', 'last_max_pad', 'last_total_pc', 'last_avg_pc']
['last_min_pc', 'last_max_pc', 'l

In [19]:
cpd.rename({'mode_product_id':'product_id'},axis=1 , inplace=True)

In [20]:
columns = cpd.columns

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 10)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

Total columns: (1410027, 112)
['customer_id', 'order_date', 'purchase_count', 'avg_quantity', 'max_quantity', 'total_spend', 'avg_spend', 'min_spend', 'max_spend', 'total_pad']
['avg_pad', 'min_pad', 'max_pad', 'total_pc', 'avg_pc', 'min_pc', 'max_pc', 'total_pd', 'avg_pd', 'min_pd']
['max_pd', 'total_pp', 'avg_pp', 'min_pp', 'max_pp', 'p25_pc', 'p50_pc', 'p75_pc', 'p90_pc', 'p95_pc']
['p25_spend', 'p50_spend', 'p75_spend', 'p90_spend', 'p95_spend', 'p25_pad', 'p50_pad', 'p75_pad', 'p90_pad', 'p95_pad']
['p25_pd', 'p50_pd', 'p75_pd', 'p90_pd', 'p95_pd', 'p25_pp', 'p50_pp', 'p75_pp', 'p90_pp', 'p95_pp']
['p25_quantity', 'p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity', 'product_id', 'last_order_date', 'last_purchase_count', 'last_avg_quantity', 'last_max_quantity']
['last_total_spend', 'last_avg_spend', 'last_min_spend', 'last_max_spend', 'last_total_pad', 'last_avg_pad', 'last_min_pad', 'last_max_pad', 'last_total_pc', 'last_avg_pc']
['last_min_pc', 'last_max_pc', 'last_t

In [21]:
cpd.to_csv("Final_Feature_code.csv")

In [ ]:
import pandas as pd
import numpy as np
sales= pd.read_csv("Final_Feature_code.csv")
sales.drop('order_date',axis=1, inplace= True)
sales.drop('customer_id',axis=1, inplace= True)
sales.drop('Unnamed: 0',axis=1, inplace= True)
sales.columns


In [ ]:
sales= sales.drop('last_order_date',axis=1)

In [ ]:
X_train= sales.drop('product_id',axis=1)
y_train = sales['product_id']

In [ ]:
import lightgbm as lgb
import shap
import numpy as np
import pandas as pd

# ============================================================
# 1. TRAIN LIGHTGBM
# ============================================================

model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

model.fit(X_train, y_train)


# ============================================================
# 2. SAMPLE DATA FOR SHAP
# ============================================================

# SHAP does NOT need every row for global feature selection
SHAP_SAMPLE_SIZE = 20000

if len(X_train) > SHAP_SAMPLE_SIZE:
    X_shap = X_train.sample(
        n=SHAP_SAMPLE_SIZE,
        random_state=42
    )
else:
    X_shap = X_train


print("Training rows :", len(X_train))
print("SHAP rows     :", len(X_shap))


# ============================================================
# 3. CALCULATE SHAP VALUES
# ============================================================

explainer = shap.TreeExplainer(model)

shap_values = explainer.shap_values(
    X_shap,
    check_additivity=False
)


# ============================================================
# 4. CONVERT SHAP OUTPUT
# ============================================================

if isinstance(shap_values, list):

    # Binary classification
    if len(shap_values) == 2:
        shap_values = shap_values[1]

    # Multiclass
    else:
        shap_values = np.stack(shap_values, axis=-1)


shap_values = np.asarray(shap_values)


# ============================================================
# 5. GLOBAL FEATURE IMPORTANCE
# ============================================================

if shap_values.ndim == 2:

    # rows × features
    importance = np.mean(
        np.abs(shap_values),
        axis=0
    )

elif shap_values.ndim == 3:

    # rows × features × classes
    importance = np.mean(
        np.abs(shap_values),
        axis=(0, 2)
    )

else:
    raise ValueError(
        f"Unexpected SHAP shape: {shap_values.shape}"
    )


importance = np.asarray(importance).ravel()


# ============================================================
# 6. CREATE IMPORTANCE DATAFRAME
# ============================================================

shap_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": importance
})

shap_df.sort_values(
    "importance",
    ascending=False,
    inplace=True,
    ignore_index=True
)


# ============================================================
# 7. IMPORTANCE %
# ============================================================

total = shap_df["importance"].sum()

if total == 0:
    raise ValueError("Total SHAP importance is 0.")

shap_df["importance_pct"] = (
    shap_df["importance"]
    .div(total)
    .mul(100)
)

shap_df["cumulative_pct"] = (
    shap_df["importance_pct"].cumsum()
)


# ============================================================
# 8. SELECT FEATURES COVERING 95%
# ============================================================

mask = shap_df["cumulative_pct"].to_numpy() >= 95

if mask.any():
    cutoff = np.argmax(mask)
else:
    cutoff = len(shap_df) - 1

top_95 = shap_df.iloc[:cutoff + 1].copy()

selected_features = top_95["feature"].tolist()


# ============================================================
# 9. RESULT
# ============================================================

print("\n" + "=" * 60)
print("SHAP FEATURE SELECTION")
print("=" * 60)

print("Total features       :", len(shap_df))
print("Selected features    :", len(selected_features))
print(
    "Cumulative importance:",
    round(top_95["cumulative_pct"].iloc[-1], 2),
    "%"
)

print("\nSelected Features:")
print(selected_features)

print("\nTop 95% SHAP Features:")

print(
    top_95[
        [
            "feature",
            "importance",
            "importance_pct",
            "cumulative_pct"
        ]
    ].to_string(index=False)
)